# Kimi

In [1]:
import os
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from huggingface_hub import snapshot_download
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

os.environ["HF_HOME"] = CACHE_DIR
LOCAL_MODEL_PATH = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)

Fetching 64 files:   0%|          | 0/64 [00:00<?, ?it/s]

In [2]:
model = KimiAudio(model_path=LOCAL_MODEL_PATH, load_detokenizer=True)
print("Kimi Audio model loaded.")

2026-03-24 13:58:50.055 | INFO     | kimia_infer.api.kimia:__init__:16 - Loading kimi-audio main model
2026-03-24 13:58:50.058 | INFO     | kimia_infer.api.kimia:__init__:25 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-03-24 13:58:50.059 | INFO     | kimia_infer.api.kimia:__init__:26 - Loading whisper model
`torch_dtype` is deprecated! Use `dtype` instead!
using normal flash attention


Loading checkpoint shards:   0%|          | 0/36 [00:00<?, ?it/s]

2026-03-24 13:58:58.556 | INFO     | kimia_infer.api.prompt_manager:__init__:20 - Looking for resources in /root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b
2026-03-24 13:58:58.558 | INFO     | kimia_infer.api.prompt_manager:__init__:21 - Loading whisper model
2026-03-24 13:58:59.495 | INFO     | kimia_infer.api.prompt_manager:__init__:30 - Loading text tokenizer
2026-03-24 13:58:59.688 | INFO     | kimia_infer.api.kimia:__init__:41 - Loading detokenizer


ninja: no work to do.


/root/autodl-tmp/envs/kimi/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Loading '/root/autodl-tmp/LLM_Model/models--moonshotai--Kimi-Audio-7B-Instruct/snapshots/9a82a84c37ad9eb1307fb6ed8d7b397862ef9e6b/vocoder/model.pt'
Complete.
using rope base theta = 10000.0, interpolation factor = 1.0
Currently using bfloat16 for PrefixFlowMatchingDetokenizer
Kimi Audio model loaded.


In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specializing in Alzheimer's disease "
    "and dementia detection from spontaneous speech. Analyze speech using clinical reasoning.\n\n"

    "Evaluate:\n"
    "- Word-finding difficulty and vague expressions\n"
    "- Semantic errors and low informational content\n"
    "- Simplified or incomplete sentence structure\n"
    "- Pauses, repetitions, and self-repair patterns\n"
    "- Coherence, logic, and topic maintenance\n"
    "- Cognitive-linguistic signs (e.g., confusion, unclear references)\n\n"

    "Note: Normal aging may include mild pauses, but dementia shows persistent, "
    "multi-domain impairment. Base your judgment on the overall pattern."
)

USER_PROMPT = (
    "Listen to the speech sample (or read the transcript) and assess the speaker.\n\n"

    "Determine whether the overall language pattern shows signs of dementia.\n\n"

    "IMPORTANT:\n"
    "- Output exactly ONE word\n"
    "- No explanation or extra text\n\n"

    "Answer:\n"
    "Dementia\n"
    "or\n"
    "Control"
)

In [4]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [5]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    messages = [
        {"role": "user", "message_type": "text",  "content": SYSTEM_PROMPT + "\n\n" + USER_PROMPT},
        {"role": "user", "message_type": "audio", "content": str(wav_path)},
    ]
    _, text = model.generate(messages, output_type="text")
    return text

In [6]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [7]:
import sys; sys.path.insert(0, str(PROJECT_ROOT / "train"))
from data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

[Pitt-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Pitt, exists=True


Pitt-raw:   0%|          | 1/551 [00:01<17:07,  1.87s/it]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-raw:   0%|          | 2/551 [00:02<09:15,  1.01s/it]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  INVALID [187] session=248-0 true=Control raw='The man is describing a scene with a child reaching for cookies, a mother doing dishes, and water overflowing from the sink.'


Pitt-raw:  75%|███████▌  | 415/551 [02:45<01:17,  1.76it/s]

  INVALID [414] session=271-2 true=Dementia raw="The woman's speech is characterized by frequent pauses, word-finding difficulties, and vague descriptions, which are indicative of dementia-related language impairment."


Pitt-raw: 100%|██████████| 551/551 [03:34<00:00,  2.56it/s]

[Pitt-raw]
  Accuracy:    0.5938
  F1:          0.7304
  Control Acc: 0.0996
  Dementia Acc:0.9805
  Valid: 549/551  Skipped: 0


In [8]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

[Lu-raw] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/raw/Lu, exists=True


Lu-raw:   1%|▏         | 1/74 [00:00<00:30,  2.39it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-raw:   3%|▎         | 2/74 [00:00<00:25,  2.83it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-raw:   4%|▍         | 3/74 [00:01<00:25,  2.83it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-raw: 100%|██████████| 74/74 [00:23<00:00,  3.16it/s]

[Lu-raw]
  Accuracy:    0.5541
  F1:          0.6733
  Control Acc: 0.1944
  Dementia Acc:0.8947
  Valid: 74/74  Skipped: 0


In [9]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

[Pitt-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Demucs, exists=True


Pitt-Demucs:   0%|          | 1/551 [00:00<03:33,  2.57it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Demucs:   0%|          | 2/551 [00:00<03:38,  2.51it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-Demucs:  75%|███████▌  | 415/551 [02:43<01:12,  1.87it/s]

  INVALID [414] session=271-2 true=Dementia raw='The woman is describing a scene involving a boy, a cookie jar, a stool, and a mother washing dishes.'


Pitt-Demucs: 100%|██████████| 551/551 [03:33<00:00,  2.58it/s]

[Pitt-Demucs]
  Accuracy:    0.5782
  F1:          0.7225
  Control Acc: 0.0661
  Dementia Acc:0.9805
  Valid: 550/551  Skipped: 0


In [10]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

[Lu-Demucs] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Demucs, exists=True


Lu-Demucs:   1%|▏         | 1/74 [00:00<00:24,  2.93it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Demucs:   3%|▎         | 2/74 [00:00<00:22,  3.22it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Demucs:   4%|▍         | 3/74 [00:00<00:23,  3.08it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Demucs: 100%|██████████| 74/74 [00:22<00:00,  3.22it/s]

[Lu-Demucs]
  Accuracy:    0.5405
  F1:          0.6792
  Control Acc: 0.1111
  Dementia Acc:0.9474
  Valid: 74/74  Skipped: 0


In [11]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

[Pitt-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Denoiser, exists=True


Pitt-Denoiser:   0%|          | 1/551 [00:00<03:23,  2.70it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Denoiser:   0%|          | 2/551 [00:00<03:20,  2.73it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  INVALID [211] session=296-0 true=Control raw='The woman is describing a scene involving a boy, a girl, and their mother in a kitchen.'


Pitt-Denoiser:  75%|███████▌  | 415/551 [02:30<01:12,  1.88it/s]

  INVALID [414] session=271-2 true=Dementia raw='The woman is describing a scene involving a little boy, a cookie jar, a stool, and a mother washing dishes in a kitchen sink.'


Pitt-Denoiser: 100%|██████████| 551/551 [03:16<00:00,  2.81it/s]

[Pitt-Denoiser]
  Accuracy:    0.5766
  F1:          0.7191
  Control Acc: 0.0792
  Dementia Acc:0.9643
  Valid: 548/551  Skipped: 0


In [12]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

[Lu-Denoiser] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Denoiser, exists=True


Lu-Denoiser:   1%|▏         | 1/74 [00:00<00:23,  3.14it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Denoiser:   3%|▎         | 2/74 [00:00<00:20,  3.45it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Denoiser:   4%|▍         | 3/74 [00:00<00:20,  3.38it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Denoiser: 100%|██████████| 74/74 [00:20<00:00,  3.63it/s]

[Lu-Denoiser]
  Accuracy:    0.5405
  F1:          0.6909
  Control Acc: 0.0556
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [13]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

[Pitt-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-FRCRN_SE, exists=True


Pitt-FRCRN_SE:   0%|          | 1/551 [00:00<03:17,  2.78it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE:   0%|          | 2/551 [00:00<03:23,  2.69it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-FRCRN_SE:  75%|███████▌  | 415/551 [02:28<01:11,  1.90it/s]

  INVALID [414] session=271-2 true=Dementia raw='A woman is describing a scene involving a little boy, a cookie jar, a stool, and a mother washing dishes in a kitchen.'


Pitt-FRCRN_SE: 100%|██████████| 551/551 [03:14<00:00,  2.84it/s]

[Pitt-FRCRN_SE]
  Accuracy:    0.5945
  F1:          0.7304
  Control Acc: 0.1033
  Dementia Acc:0.9805
  Valid: 550/551  Skipped: 0


In [14]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

[Lu-FRCRN_SE] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-FRCRN_SE, exists=True


Lu-FRCRN_SE:   1%|▏         | 1/74 [00:00<00:22,  3.26it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   3%|▎         | 2/74 [00:00<00:20,  3.43it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-FRCRN_SE:   4%|▍         | 3/74 [00:00<00:21,  3.38it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-FRCRN_SE: 100%|██████████| 74/74 [00:20<00:00,  3.66it/s]

[Lu-FRCRN_SE]
  Accuracy:    0.5946
  F1:          0.7115
  Control Acc: 0.1944
  Dementia Acc:0.9737
  Valid: 74/74  Skipped: 0


In [15]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

[Pitt-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-MossFormer, exists=True


Pitt-MossFormer:   0%|          | 1/551 [00:00<03:28,  2.64it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-MossFormer:   0%|          | 2/551 [00:00<03:22,  2.71it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


Pitt-MossFormer:   1%|          | 3/551 [00:01<03:20,  2.74it/s]

  DEBUG [2] session=002-2 raw='Dementia' pred=Dementia


Pitt-MossFormer: 100%|██████████| 551/551 [03:13<00:00,  2.84it/s]


[Pitt-MossFormer]
  Accuracy:    0.5826
  F1:          0.7268
  Control Acc: 0.0620
  Dementia Acc:0.9903
  Valid: 551/551  Skipped: 0


In [16]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

[Lu-MossFormer] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-MossFormer, exists=True


Lu-MossFormer:   1%|▏         | 1/74 [00:00<00:24,  3.03it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-MossFormer:   3%|▎         | 2/74 [00:00<00:21,  3.37it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-MossFormer:   4%|▍         | 3/74 [00:00<00:21,  3.34it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-MossFormer: 100%|██████████| 74/74 [00:20<00:00,  3.63it/s]

[Lu-MossFormer]
  Accuracy:    0.5541
  F1:          0.6972
  Control Acc: 0.0833
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0


In [17]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

[Pitt-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Pitt-Resemble, exists=True


Pitt-Resemble:   0%|          | 1/551 [00:00<03:40,  2.50it/s]

  DEBUG [0] session=002-0 raw='Dementia' pred=Dementia


Pitt-Resemble:   0%|          | 2/551 [00:00<03:43,  2.45it/s]

  DEBUG [1] session=002-1 raw='Dementia' pred=Dementia


...

  INVALID [187] session=248-0 true=Control raw='The man is describing a scene with a child reaching for cookies, a mother doing dishes, and water overflowing from the sink.'


Pitt-Resemble:  75%|███████▌  | 415/551 [02:47<01:22,  1.65it/s]

  INVALID [414] session=271-2 true=Dementia raw='The woman is describing a scene involving a little boy, a cookie jar, and a mother washing dishes, with a focus on the details of the situation.'


Pitt-Resemble: 100%|██████████| 551/551 [03:37<00:00,  2.53it/s]

[Pitt-Resemble]
  Accuracy:    0.5811
  F1:          0.7242
  Control Acc: 0.0705
  Dementia Acc:0.9805
  Valid: 549/551  Skipped: 0


In [18]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")

[Lu-Resemble] audio_dir=/root/autodl-tmp/Few-Shot_is_all_you_need/ad_detection/data/denoised/Lu-Resemble, exists=True


Lu-Resemble:   1%|▏         | 1/74 [00:00<00:24,  3.00it/s]

  DEBUG [0] session=F22_000 raw='Dementia' pred=Dementia


Lu-Resemble:   3%|▎         | 2/74 [00:00<00:22,  3.22it/s]

  DEBUG [1] session=F22_001 raw='Dementia' pred=Dementia


Lu-Resemble:   4%|▍         | 3/74 [00:00<00:22,  3.11it/s]

  DEBUG [2] session=F26_000 raw='Dementia' pred=Dementia


Lu-Resemble: 100%|██████████| 74/74 [00:22<00:00,  3.32it/s]

[Lu-Resemble]
  Accuracy:    0.5405
  F1:          0.6909
  Control Acc: 0.0556
  Dementia Acc:1.0000
  Valid: 74/74  Skipped: 0
